# Stackelberg (Rust) : la performativité sans mystère
**Navigation** : [<< 9-BackwardInduction](GameTheory-09-BackwardInduction-Python.ipynb) | [Index](README.md) | [9c-Stackelberg-SecurityGame >>](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) | [Jumeau Python](GameTheory-09b-Commitment-Stackelberg-Python.ipynb) | [Jumeau C#](GameTheory-09b-Commitment-Stackelberg-CSharp.ipynb)


Une action **inférieure contre une réponse fixée** dans le jeu simultané peut devenir la meilleure si l'agent peut **s'y engager** avant que l'autre ne choisisse — parce que l'engagement **transforme la meilleure réponse de l'autre**. C'est le cœur de la solution de Stackelberg [von Stackelberg, 1934], et c'est aussi, déguisé en théorie des jeux, un noyau mathématique propre pour ce qu'Austin [1962] appelle le **performatif** : un énoncé (ou un acte) qui ne décrit pas le monde mais le *fait*.

```text
sigma_L  ->  BR_F(sigma_L)  ->  u_L(sigma_L, BR_F(sigma_L))
```

> **L'acte n'a plus seulement une valeur par son RÉSULTAT DIRECT. Il a une valeur parce qu'il MODIFIE L'ESPACE DE RÉPONSE D'AUTRUI.**

La condition de félicité d'Austin — ce qui distingue une annonce qui *fait* d'une annonce qui ne *dit* rien — se traduit ici en termes brutaux : **l'engagement doit être crédible**. S'il ne l'est pas, rien ne change. S'il l'est, l'annonce devient **causalement constitutive** du jeu qui suit :

```text
representation publique  ->  anticipation collective  ->  changement des actions  ->  nouvelle realite strategique
```

Ce notebook démonte la mécanique sur un jeu d'entrée sur marché, en quatre régimes calculés exactement : le jeu simultané, la mise séquentielle sans engagement, l'engagement contraignant, et l'engagement révocable — puis il **mesure** ce que coûte la crédibilité, et montre que le seuil à payer est exactement l'écart de tentation du déviateur. Une performativité sans mystère : chaque affirmation de ce notebook est un nombre calculé dans une cellule.

Le fil suivra la série [GameTheory](GameTheory-02-NormalForm-Python.ipynb) (jeux sous forme normale, [équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb), [induction à rebours](GameTheory-09-BackwardInduction-Python.ipynb)) et rejoint le chantier « strate 7 » (vocabulaires et extensions) : l'engagement est l'archétype de l'acte qui change l'espace des actions disponibles plutôt que d'en choisir une.

**Objectifs** : calculer meilleures réponses et engagements, distinguer annonce et lien, vérifier un certificat de retenue locale. **Prérequis** : formes normales et induction à rebours. **Durée estimée** : 60–90 minutes.

Rust standard uniquement (`std`, aucune dependance) ; les tableaux et arbres sont calculés depuis les matrices.


In [1]:
const ACTIONS_E: [&str; 2] = ["In", "Out"];
const ACTIONS_I: [&str; 2] = ["Acc", "Fight"];
/// Table des gains du jeu d'entree sur marche, en ordre declare.
fn table_u() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("In", "Fight"), (-1, -2)),
        (("Out", "Acc"), (0, 0)),
        (("Out", "Fight"), (0, 3)),
    ]
}
/// Gain de la case (ae, ai), lu dans la table (invariant : la case existe).
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Gain de l'entrant.
fn u_e(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).0 }
/// Gain de l'incumbent.
fn u_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).1 }
/// Meilleure reponse de l'entrant : le PREMIER maximum (l'argmax de Python garde le premier).
fn br_e(table: &[((&'static str, &'static str), (i64, i64))], ai: &str) -> &'static str {
    let mut best = ACTIONS_E[0];
    let mut best_v = u_e(table, best, ai);
    for &ae in ACTIONS_E.iter().skip(1) {
        let v = u_e(table, ae, ai);
        if v > best_v { best = ae; best_v = v; }
    }
    best
}
/// Meilleure reponse de l'incumbent : le PREMIER maximum.
fn br_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str) -> &'static str {
    let mut best = ACTIONS_I[0];
    let mut best_v = u_i(table, ae, best);
    for &ai in ACTIONS_I.iter().skip(1) {
        let v = u_i(table, ae, ai);
        if v > best_v { best = ai; best_v = v; }
    }
    best
}
/// Repr Python d'un tuple d'entiers.
fn tup2i(a: i64, b: i64) -> String { format!("({}, {})", a, b) }

let u = table_u();

println!("Matrice de gains (u_entrant, u_incumbent) :");
println!("{:10} {:>12} {:>12}", "", "Acc", "Fight");
for &ae in ACTIONS_E.iter() {
    let mut colonnes: Vec<String> = Vec::new();
    for &ai in ACTIONS_I.iter() {
        let g = gain(&u, ae, ai);
        colonnes.push(format!("{:>12}", tup2i(g.0, g.1)));
    }
    println!("{:10} {}", ae, colonnes.join(" "));
}
println!();
println!("Meilleures reponses :");
for &ai in ACTIONS_I.iter() {
    println!("  BR_entrant({:5}) = {:4}   (gains E : In={:>2}, Out={:>2})",
             ai, br_e(&u, ai), u_e(&u, "In", ai), u_e(&u, "Out", ai));
}
for &ae in ACTIONS_E.iter() {
    println!("  BR_incumbent({:4}) = {:5}   (gains I : Acc={:>2}, Fight={:>2})",
             ae, br_i(&u, ae), u_i(&u, ae, "Acc"), u_i(&u, ae, "Fight"));
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc.

Matrice de gains (u_entrant, u_incumbent) :


                    Acc        Fight


In               (2, 1)     (-1, -2)


Out              (0, 0)       (0, 3)


Meilleures reponses :


  BR_entrant(Acc  ) = In     (gains E : In= 2, Out= 0)


  BR_entrant(Fight) = Out    (gains E : In=-1, Out= 0)


  BR_incumbent(In  ) = Acc     (gains I : Acc= 1, Fight=-2)


  BR_incumbent(Out ) = Fight   (gains I : Acc= 0, Fight= 3)


### Lecture : la valeur de Fight vit dans la ligne Out

Regardez les deux meilleures réponses de l'incumbent : contre une **entrée**, il accommode (1 > −2) ; contre l'**abstention**, il « combat » (3 > 0) — combattre un marché vide ne coûte rien et laisse le monopole intact. L'action `Fight` n'est donc pas dominée dans l'absolu, mais elle est **strictement inférieure contre cette entrée** : sur la ligne `In`, elle vaut −2 contre 1 pour `Acc`. Toute sa valeur (3) vit dans la ligne `Out` — une colonie de gains qui n'existe que si l'entrant reste dehors.

C'est la tension de tout le notebook : `Fight` est l'action qu'il *faudrait* pouvoir choisir, précisément dans le monde où elle ne coûte rien — et l'entrant le sait. La question de Stackelberg est de savoir si l'incumbent peut **faire exister** la ligne `Out`. La question d'Austin est de savoir à quel prix l'annonce « je combattrai » devient un fait plutôt qu'un souffle.

In [2]:
const ACTIONS_E: [&str; 2] = ["In", "Out"];
const ACTIONS_I: [&str; 2] = ["Acc", "Fight"];
/// Table des gains du jeu d'entree sur marche, en ordre declare.
fn table_u() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("In", "Fight"), (-1, -2)),
        (("Out", "Acc"), (0, 0)),
        (("Out", "Fight"), (0, 3)),
    ]
}
/// Gain de la case (ae, ai), lu dans la table (invariant : la case existe).
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Gain de l'entrant.
fn u_e(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).0 }
/// Gain de l'incumbent.
fn u_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).1 }
/// Meilleure reponse de l'entrant : le PREMIER maximum (l'argmax de Python garde le premier).
fn br_e(table: &[((&'static str, &'static str), (i64, i64))], ai: &str) -> &'static str {
    let mut best = ACTIONS_E[0];
    let mut best_v = u_e(table, best, ai);
    for &ae in ACTIONS_E.iter().skip(1) {
        let v = u_e(table, ae, ai);
        if v > best_v { best = ae; best_v = v; }
    }
    best
}
/// Meilleure reponse de l'incumbent : le PREMIER maximum.
fn br_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str) -> &'static str {
    let mut best = ACTIONS_I[0];
    let mut best_v = u_i(table, ae, best);
    for &ai in ACTIONS_I.iter().skip(1) {
        let v = u_i(table, ae, ai);
        if v > best_v { best = ai; best_v = v; }
    }
    best
}

let u = table_u();

// Equilibres de Nash en actions pures : les deux filtres, sur les deux tables.
let mut nes: Vec<(&'static str, &'static str)> = Vec::new();
for &ae in ACTIONS_E.iter() {
    for &ai in ACTIONS_I.iter() {
        if ae == br_e(&u, ai) && ai == br_i(&u, ae) {
            nes.push((ae, ai));
        }
    }
}
println!("Equilibres de Nash (actions pures) du jeu simultane :");
for (ae, ai) in nes.iter() {
    println!("  ({}, {})  ->  u_E = {:>2}, u_I = {:>2}", ae, ai, u_e(&u, ae, ai), u_i(&u, ae, ai));
}

println!();
println!("Mise sequentielle (entrant puis incumbent), induction a rebours :");
for &ae in ACTIONS_E.iter() {
    println!("  si E joue {:4} : I repond {:5} -> u_E = {:>2}", ae, br_i(&u, ae), u_e(&u, ae, br_i(&u, ae)));
}
// Issue sequentielle : le PREMIER maximum du gain de l'entrant.
let mut seq_ae = ACTIONS_E[0];
let mut seq_v = i64::MIN;
for &ae in ACTIONS_E.iter() {
    let v = u_e(&u, ae, br_i(&u, ae));
    if v > seq_v { seq_v = v; seq_ae = ae; }
}
println!("  -> issue sequentielle : ({}, {}), u_E = {}, u_I = {}",
         seq_ae, br_i(&u, seq_ae), u_e(&u, seq_ae, br_i(&u, seq_ae)), u_i(&u, seq_ae, br_i(&u, seq_ae)));

Equilibres de Nash (actions pures) du jeu simultane :


  (In, Acc)  ->  u_E =  2, u_I =  1


  (Out, Fight)  ->  u_E =  0, u_I =  3


Mise sequentielle (entrant puis incumbent), induction a rebours :


  si E joue In   : I repond Acc   -> u_E =  2


  si E joue Out  : I repond Fight -> u_E =  0


  -> issue sequentielle : (In, Acc), u_E = 2, u_I = 1


### Lecture : deux équilibres simultanés, un seul survit à la séquentialisation

Le jeu simultané porte **deux** équilibres de Nash : `(In, Acc)` où l'entrant vient et se fait accueillir (l'incumbent touche 1), et `(Out, Fight)` où l'entrant reste dehors face à la menace (l'incumbent touche 3). Le second vaut trois fois le premier pour l'incumbent — mais regardez ce que la mise séquentielle en fait : dès que l'entrant peut *tester* la menace en jouant le premier, l'induction à rebours la dissout. L'incumbent qui fait face à une entrée réelle accommode (1 > −2), l'entrant le sait, entre, et l'issue retombe sur `(In, Acc)` : **u_I = 1**.

L'équilibre `(Out, Fight)` n'était pas faux — c'est un équilibre de Nash parfaitement valide de la forme normale — il était **incroyable** : sa valeur repose sur une action (`Fight` face à `In`) que l'incumbent ne choisirait jamais au moment de la choisir. La différence 3 − 1 = **2** n'est pas perdue : elle est *en attente*. C'est exactement le montant qu'un engagement contraignant peut débloquer dans cette comparaison de mécanismes.

In [3]:
const ACTIONS_E: [&str; 2] = ["In", "Out"];
const ACTIONS_I: [&str; 2] = ["Acc", "Fight"];
/// Table des gains du jeu d'entree sur marche, en ordre declare.
fn table_u() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("In", "Fight"), (-1, -2)),
        (("Out", "Acc"), (0, 0)),
        (("Out", "Fight"), (0, 3)),
    ]
}
/// Gain de la case (ae, ai), lu dans la table (invariant : la case existe).
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Gain de l'entrant.
fn u_e(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).0 }
/// Gain de l'incumbent.
fn u_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).1 }
/// Meilleure reponse de l'entrant : le PREMIER maximum (l'argmax de Python garde le premier).
fn br_e(table: &[((&'static str, &'static str), (i64, i64))], ai: &str) -> &'static str {
    let mut best = ACTIONS_E[0];
    let mut best_v = u_e(table, best, ai);
    for &ae in ACTIONS_E.iter().skip(1) {
        let v = u_e(table, ae, ai);
        if v > best_v { best = ae; best_v = v; }
    }
    best
}
/// Meilleure reponse de l'incumbent : le PREMIER maximum.
fn br_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str) -> &'static str {
    let mut best = ACTIONS_I[0];
    let mut best_v = u_i(table, ae, best);
    for &ai in ACTIONS_I.iter().skip(1) {
        let v = u_i(table, ae, ai);
        if v > best_v { best = ai; best_v = v; }
    }
    best
}
/// Type rationnel du carnet : numerateur / denominateur normalises, denominateur > 0.
#[derive(Clone, Copy, PartialEq, Eq)]
struct Rat { n: i64, d: i64 }

/// PGCD par l'algorithme d'Euclide.
fn pgcd(mut a: i64, mut b: i64) -> i64 {
    while b != 0 { let t = a % b; a = b; b = t; }
    a
}

impl Rat {
    fn new(n: i64, d: i64) -> Rat {
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = pgcd(n.abs(), d);
        Rat { n: n / g, d: d / g }
    }
    fn sub(&self, o: &Rat) -> Rat { Rat::new(self.n * o.d - o.n * self.d, self.d * o.d) }
    fn mul(&self, o: &Rat) -> Rat { Rat::new(self.n * o.n, self.d * o.d) }
    fn abs(&self) -> Rat { Rat::new(self.n.abs(), self.d) }
    fn eq_rat(&self, o: &Rat) -> bool { self.n * o.d == o.n * self.d }
    fn lt(&self, o: &Rat) -> bool { self.n * o.d < o.n * self.d }
    fn gt(&self, o: &Rat) -> bool { self.n * o.d > o.n * self.d }
    fn to_f64(&self) -> f64 { self.n as f64 / self.d as f64 }
}

impl std::fmt::Display for Rat {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        // `f.pad` : c'est lui qui applique la largeur et l'alignement demandes
        // par le format (`{: <10}`) ; un `write!` interne les perdrait.
        if self.d == 1 { f.pad(&self.n.to_string()) } else { f.pad(&format!("{}/{}", self.n, self.d)) }
    }
}

let u = table_u();

// Issue sequentielle (cellule autonome) : l'engagement se mesure contre elle.
let mut seq_ae = ACTIONS_E[0];
let mut seq_v = i64::MIN;
for &ae in ACTIONS_E.iter() {
    let v = u_e(&u, ae, br_i(&u, ae));
    if v > seq_v { seq_v = v; seq_ae = ae; }
}

println!("Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)");
// Ordre d'affichage : celui de ACTIONS_I (le dict Python garde l'ordre d'insertion).
let mut best_ai = ACTIONS_I[0];
let mut best_gain = i64::MIN;
for &ai in ACTIONS_I.iter() {
    let r = br_e(&u, ai);
    let g = u_i(&u, r, ai);
    println!("  s engager a {:5} -> E repond {:4} -> u_I = {:>2}", ai, r, g);
    if g > best_gain { best_gain = g; best_ai = ai; }
}
let gain_seq = u_i(&u, seq_ae, br_i(&u, seq_ae));
println!("  -> engagement optimal : {} (u_I = {}), contre {} sans engagement : delta = {:+}",
         best_ai, best_gain, gain_seq, best_gain - gain_seq);

println!();
println!("Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I");
println!("{:>5} {:>6} {:>8}", "p", "BR_E", "u_I");
for k in 0..21 {
    let p = k as f64 / 20.0;
    let in_pay = 2.0 - 3.0 * p;
    let r = if in_pay > 0.0 { "In" } else if in_pay < 0.0 { "Out" } else { "indifferent" };
    let val = if r == "In" { 1.0 - 3.0 * p } else { 3.0 * p };
    println!("{:>5.2} {:>10} {:>8.2}", p, r, val);
}
// Le seuil exact : une fraction, pas un flottant.
let seuil_p = Rat::new(2, 3);
let favorable = Rat::new(3, 1).mul(&seuil_p);
let defavorable = Rat::new(1, 1).sub(&favorable);
println!("Au seuil exact p={} : BR_E=In ou Out ; u_I favorable={}, défavorable={}",
         seuil_p, favorable, defavorable);

Engagement contraignant : tableau (engagement -> reponse de E -> gain de I)


  s engager a Acc   -> E repond In   -> u_I =  1


  s engager a Fight -> E repond Out  -> u_I =  3


  -> engagement optimal : Fight (u_I = 3), contre 1 sans engagement : delta = +2


Engagement mixte (p = P(Fight)) : meilleure reponse de E et gain de I


    p   BR_E      u_I


... (18 outputs left out)

 0.90        Out     2.70


 0.95        Out     2.85


 1.00        Out     3.00


Au seuil exact p=2/3 : BR_E=In ou Out ; u_I favorable=2, défavorable=-1


### Lecture : l'action sous-optimale devient l'engagement unique optimal

Le tableau de l'engagement est la cellule centrale du notebook :

| engagement de I | réponse de E | gain de I |
|---|---|---|
| `Acc` | `In` | 1 |
| **`Fight`** | **`Out`** | **3** |

S'engager à `Acc` ne change rien (l'entrant vient, gain 1). S'engager à `Fight` **transforme la meilleure réponse de l'entrant** — face à un adversaire lié à la guerre de prix, `Out` (0) domine `In` (−1) — et rapporte 3. Le delta est **+2**, exactement l'écart entre les deux équilibres du jeu simultané : l'engagement n'invente pas de valeur, il **sélectionne** l'équilibre que la menace incroyable ne pouvait pas tenir.

Et le point décisif, celui qui donne son titre à l'issue : `Fight` était **strictement inférieure contre une entrée** (−2 < 1 sur la ligne `In`). Elle devient l'engagement *unique* optimal — non parce que sa valeur directe aurait changé (combattre une entrée vaut toujours −2), mais parce que **commettre la transforme en device de sélection** : l'acte ne vaut pas par son résultat direct, il vaut par la modification de l'espace de réponse d'autrui. C'est le sens exact de la chaîne `sigma_L -> BR_F(sigma_L) -> u_L` ouverte en tête de notebook.

L'engagement **mixte** referme la porte d'une évasion : aucun mixage ne bat le sommet pur. Tant que p < 2/3, l'entrant entre et le gain de l'incumbent décroît (1 − 3p) ; pour p > 2/3, l'entrant sort et le gain plafonne à 3p, maximal à p = 1. À p = 2/3, les deux réponses de l’entrant sont optimales : la sélection favorable au leader donne 2, la défavorable −1. Le balayage au pas 0,05 ne contient pas ce seuil ; il est calculé séparément. La valeur d'engagement de ce jeu est un **sommet du simplexe**, pas un intérieur — la nuance honnête étant qu'il existe d'autres jeux (les jeux d'inspection, où le contrôleur doit rester imprévisible) où c'est l'intérieur qui gagne : le phénomène « mixer bat commettre » existe, ce n'est simplement pas celui d'ici.

La sélection d’un Nash existant est un résultat de cette matrice ; elle ne caractérise pas tous les jeux d’engagement. Le duel publicitaire ci-dessous présente déjà un meilleur engagement mixte intérieur.



## Exercice 1 : le duel publicitaire — équilibres et action candidate
<a id="duel-publicitaire"></a>

Le jeu de ce notebook n'est pas le seul où vit un engagement. Soit le duel publicitaire suivant : deux firmes A (leader potentiel) et B ; A choisit `PasCampagne` / `Campagne`, B choisit `Reste` / `Attaque`. Gains `(u_A, u_B)` : `('PasCampagne','Reste'): (3, 3)`, `('PasCampagne','Attaque'): (1, 4)`, `('Campagne','Reste'): (2, 0)`, `('Campagne','Attaque'): (0, -1)`.

Calculez les meilleures réponses, listez **tous** les équilibres de Nash en actions pures du jeu simultané, puis l'issue de la mise séquentielle où B joue le premier. Identifiez enfin l'action de A qui est sous-optimale conditionnellement à `Attaque` — la candidate au rôle que `Fight` joue ci-dessus.

La fonction étudiante reçoit la matrice. Retourner les correspondances complètes `br_a`, `br_b`, `nash`, `b_premier` et `candidate`. `candidate` liste les actions strictement inférieures de A contre Attaque. Pour B-premier, les égalités de A sont départagées en faveur de B, puis toutes les issues maximales de B sont rendues. Le contrôle utilise aussi une matrice modifiée et une égalité : une réponse constante ne suffit pas.

```rust
# TODO etudiant
# Etape 1 : coder la matrice U_EXO1 et les correspondances br_a / br_b.
# Etape 2 : énumérer tous les équilibres de Nash.
# Etape 3 : induction à rebours B-premier et action candidate.
```

In [4]:
type DuelCase = ((&'static str, &'static str), (i64, i64));

const AA: [&str; 2] = ["PasCampagne", "Campagne"];
const BB: [&str; 2] = ["Reste", "Attaque"];

/// Gain de la case (a, b), lu dans la matrice.
fn gain_d(m: &[DuelCase], a: &str, b: &str) -> (i64, i64) {
    m.iter().find(|(k, _)| k.0 == a && k.1 == b).map(|(_, v)| *v).expect("case de la matrice")
}

/// Matrice du duel publicitaire.
fn u_exo1() -> Vec<DuelCase> {
    vec![
        (("PasCampagne", "Reste"), (3, 3)),
        (("PasCampagne", "Attaque"), (1, 4)),
        (("Campagne", "Reste"), (2, 0)),
        (("Campagne", "Attaque"), (0, -1)),
    ]
}

/// Reference de B-premier : A repond par le max lexicographique (u_A, u_B), puis
/// toutes les issues de B maximales sont rendues.
fn reference_b_premier(matrice: &[DuelCase]) -> Vec<(&'static str, &'static str)> {
    let mut reponses: Vec<(&'static str, &'static str)> = Vec::new();
    for &b in BB.iter() {
        let mut meilleure = AA[0];
        let mut meilleur = gain_d(matrice, meilleure, b);
        for &a in AA.iter().skip(1) {
            let g = gain_d(matrice, a, b);
            if (g.0, g.1) > (meilleur.0, meilleur.1) { meilleure = a; meilleur = g; }
        }
        reponses.push((meilleure, b));
    }
    let cible = reponses.iter().map(|(a, b)| gain_d(matrice, *a, *b).1).fold(i64::MIN, i64::max);
    reponses.into_iter().filter(|(a, b)| gain_d(matrice, *a, *b).1 == cible).collect()
}
use std::collections::BTreeSet;

/// Contrat de retour de l'etudiant : les cinq correspondances du carnet.
#[allow(dead_code)]
#[derive(Clone)]
struct DuelRep {
    br_a: Vec<(&'static str, Vec<&'static str>)>,
    br_b: Vec<(&'static str, Vec<&'static str>)>,
    nash: Vec<(&'static str, &'static str)>,
    b_premier: Vec<(&'static str, &'static str)>,
    candidate: Vec<&'static str>,
}

fn analyser_duel(_matrice: &[DuelCase]) -> Option<DuelRep> {
    // TODO etudiant : dictionnaire br_a, br_b, nash, b_premier, candidate.
    // Etape 1 : coder la matrice U_EXO1 et les correspondances br_a / br_b.
    // Etape 2 : enumerer tous les equilibres de Nash.
    // Etape 3 : induction a rebours B-premier et action candidate.
    None
}

let base = u_exo1();
let f_plus: Vec<DuelCase> = base.iter().map(|((a, b), v)| {
    let bonus = if (*a, *b) == ("Campagne", "Attaque") { 3 } else { 0 };
    ((*a, *b), (v.0 + bonus, v.1))
}).collect();
let f_zero: Vec<DuelCase> = base.iter().map(|((a, b), v)| ((*a, *b), (v.0, 0))).collect();

let mut complet = true;
for matrice in [base.clone(), f_plus.clone(), f_zero.clone()].iter() {
    match analyser_duel(matrice) {
        None => {
            println!("Exercice 1 à compléter : matrice, BR, Nash et B-premier");
            complet = false;
            break;
        }
        Some(r) => {
            // br_a : toutes les actions de A qui maximisent u_A, colonne par colonne.
            for &b in BB.iter() {
                let maxi = AA.iter().map(|x| gain_d(matrice, *x, b).0).fold(i64::MIN, i64::max);
                let attendu: BTreeSet<&'static str> = AA.iter().filter(|x| gain_d(matrice, **x, b).0 == maxi).copied().collect();
                let obtenu: BTreeSet<&'static str> = r.br_a.iter().find(|(k, _)| *k == b)
                    .map(|(_, v)| v.iter().copied().collect()).unwrap_or_default();
                assert!(attendu == obtenu, "br_a incorrect sur la colonne {}", b);
            }
            // br_b : de meme, ligne par ligne.
            for &a in AA.iter() {
                let maxi = BB.iter().map(|y| gain_d(matrice, a, *y).1).fold(i64::MIN, i64::max);
                let attendu: BTreeSet<&'static str> = BB.iter().filter(|y| gain_d(matrice, a, **y).1 == maxi).copied().collect();
                let obtenu: BTreeSet<&'static str> = r.br_b.iter().find(|(k, _)| *k == a)
                    .map(|(_, v)| v.iter().copied().collect()).unwrap_or_default();
                assert!(attendu == obtenu, "br_b incorrect sur la ligne {}", a);
            }
            // nash : les couples mutuellement best-reponse (les egalites sont conservees).
            let mut attendu_nash: BTreeSet<(&'static str, &'static str)> = BTreeSet::new();
            for &a in AA.iter() {
                for &b in BB.iter() {
                    let max_a = AA.iter().map(|x| gain_d(matrice, *x, b).0).fold(i64::MIN, i64::max);
                    let max_b = BB.iter().map(|y| gain_d(matrice, a, *y).1).fold(i64::MIN, i64::max);
                    let g = gain_d(matrice, a, b);
                    if g.0 == max_a && g.1 == max_b { attendu_nash.insert((a, b)); }
                }
            }
            let obtenu_nash: BTreeSet<(&'static str, &'static str)> = r.nash.iter().copied().collect();
            assert!(attendu_nash == obtenu_nash, "nash incorrect");
            // b_premier : compare a la reference declaree.
            let attendu_bp: BTreeSet<(&'static str, &'static str)> = reference_b_premier(matrice).into_iter().collect();
            let obtenu_bp: BTreeSet<(&'static str, &'static str)> = r.b_premier.iter().copied().collect();
            assert!(attendu_bp == obtenu_bp, "b_premier incorrect");
            // candidate : les actions de A strictement inferieures contre Attaque.
            let maxi_attaque = AA.iter().map(|x| gain_d(matrice, *x, "Attaque").0).fold(i64::MIN, i64::max);
            let attendu_cand: BTreeSet<&'static str> = AA.iter()
                .filter(|x| gain_d(matrice, **x, "Attaque").0 < maxi_attaque).copied().collect();
            let obtenu_cand: BTreeSet<&'static str> = r.candidate.iter().copied().collect();
            assert!(attendu_cand == obtenu_cand, "candidate incorrect");
        }
    }
}
if complet {
    println!("Exercice 1 : BR, Nash, B-premier et action candidate validés sur trois matrices");
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc.

Exercice 1 à compléter : matrice, BR, Nash et B-premier


## Annonce révocable : induction dans les sous-arbres

L’annonce n’enlève ici aucune action au dernier décideur. Calculons sa meilleure réponse puis celle du premier joueur.

In [5]:
const ACTIONS_E: [&str; 2] = ["In", "Out"];
const ACTIONS_I: [&str; 2] = ["Acc", "Fight"];
/// Table des gains du jeu d'entree sur marche, en ordre declare.
fn table_u() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("In", "Fight"), (-1, -2)),
        (("Out", "Acc"), (0, 0)),
        (("Out", "Fight"), (0, 3)),
    ]
}
/// Gain de la case (ae, ai), lu dans la table (invariant : la case existe).
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Gain de l'incumbent.
fn u_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).1 }
/// Meilleure reponse de l'incumbent : le PREMIER maximum.
fn br_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str) -> &'static str {
    let mut best = ACTIONS_I[0];
    let mut best_v = u_i(table, ae, best);
    for &ai in ACTIONS_I.iter().skip(1) {
        let v = u_i(table, ae, ai);
        if v > best_v { best = ai; best_v = v; }
    }
    best
}
/// Repr Python d'un tuple d'entiers.
fn tup2i(a: i64, b: i64) -> String { format!("({}, {})", a, b) }

let u = table_u();

// Arbre a trois etages : I annonce (sans effet lie), E choisit, I choisit LIBREMENT.
// Toutes les annonces menent au meme sous-arbre : seule compte la decision finale de I.
fn sous_arbre_final(table: &[((&'static str, &'static str), (i64, i64))], ae: &str) -> Vec<(&'static str, (i64, i64))> {
    ACTIONS_I.iter().map(|&ai| (ai, gain(table, ae, ai))).collect()
}

/// Induction a rebours : etage du bas sur u_I, etage du haut sur u_E.
fn induction_rebours(arbre_bas: &[(&'static str, Vec<(&'static str, (i64, i64))>)]) -> (&'static str, (&'static str, (i64, i64))) {
    let mut bas: Vec<(&'static str, (&'static str, (i64, i64)))> = Vec::new();
    for (ae, sous) in arbre_bas.iter() {
        let mut best = sous[0];
        for &candidat in sous.iter().skip(1) {
            if candidat.1 .1 > best.1 .1 { best = candidat; }
        }
        bas.push((ae, best));
    }
    let mut issue = bas[0];
    for &candidat in bas.iter().skip(1) {
        if candidat.1 .1 .0 > issue.1 .1 .0 { issue = candidat; }
    }
    issue
}

let mut arbre: Vec<(&'static str, Vec<(&'static str, (i64, i64))>)> = Vec::new();
for &ae in ACTIONS_E.iter() {
    arbre.push((ae, sous_arbre_final(&u, ae)));
}
let (action_E, (action_I, payoffs)) = induction_rebours(&arbre);

println!("Annonce revocable (aucun lien), induction a rebours sur tout l arbre :");
for &ae in ACTIONS_E.iter() {
    let mut rep_i = ACTIONS_I[0];
    let mut rep_v = i64::MIN;
    for &ai in ACTIONS_I.iter() {
        let v = u_i(&u, ae, ai);
        if v > rep_v { rep_v = v; rep_i = ai; }
    }
    let g = gain(&u, ae, rep_i);
    println!("  si E joue {:4} : I choisit en dernier {:5} -> {}", ae, rep_i, tup2i(g.0, g.1));
}
println!("  -> issue : ({}, {})  ->  u_E = {}, u_I = {}", action_E, action_I, payoffs.0, payoffs.1);
println!("  -> le gain d engagement (3) retombe a {} : dissous, quelle que soit l annonce.", payoffs.1);

Annonce revocable (aucun lien), induction a rebours sur tout l arbre :


  si E joue In   : I choisit en dernier Acc   -> (2, 1)


  si E joue Out  : I choisit en dernier Fight -> (0, 3)


  -> issue : (In, Acc)  ->  u_E = 2, u_I = 1


  -> le gain d engagement (3) retombe a 1 : dissous, quelle que soit l annonce.


### Lecture : la condition de félicité — l'annonce n'est pas l'engagement

Retirez le lien (l'irrévocabilité) et tout s'effondre dans l'ordre exact prédit par la théorie : au dernier étage, l'incumbent face à une entrée **réelle** accommode (1 > −2) — au moment de choisir, la menace n'a plus de valeur ; l'entrant, qui fait cette induction à rebours autant que nous, entre (2 > 0) ; et le gain de l'engagement **3 → 1**. Aucune annonce, si forte soit-elle, n'y change rien : l'issue est **indépendante de l'annonce**. Cette indépendance de l’annonce est calculée dans ce jeu à information commune et aux sous-arbres identiques ; elle n’est pas un théorème général d’inutilité de toute communication.

C'est ici que la condition de félicité d'Austin cesse d'être une métaphore littéraire. « Je combattrai » n'est pas un énoncé performatif en soi ; il ne le devient que si l'énonciation **modifie les gains ou les contraintes du locuteur** — si se dédire coûte. L'annonce révocable est un énoncé *constatif* déguisé : elle décrit une intention que tout le monde sait révisable, et l'anticipation collective la traite comme telle. La performativité n'est pas dans les mots : elle est dans **l'architecture du jeu que les mots modifient**.

## Caution : pénaliser la déviation

Le contrat retire s au gain d’Acc après une promesse Fight. Le balayage garde les neuf montants de 0 à 4, par pas de 0,5.

In [6]:
const ACTIONS_E: [&str; 2] = ["In", "Out"];
const ACTIONS_I: [&str; 2] = ["Acc", "Fight"];
/// Table des gains du jeu d'entree sur marche, en ordre declare.
fn table_u() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("In", "Fight"), (-1, -2)),
        (("Out", "Acc"), (0, 0)),
        (("Out", "Fight"), (0, 3)),
    ]
}
/// Gain de la case (ae, ai), lu dans la table (invariant : la case existe).
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Gain de l'incumbent.
fn u_i(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> i64 { gain(table, ae, ai).1 }
/// Repr Python d'un couple de chaines (apostrophes simples).
fn tup2s(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }

let u = table_u();

// L'incumbent signe un contrat : accommoder APRES avoir promis de combattre coute s.
// Payoff modifie : u_I(In, Acc) = 1 - s. Le reste est inchange.
fn equilibre_avec_bond(table: &[((&'static str, &'static str), (i64, i64))], s: f64) -> ((&'static str, &'static str), (f64, f64)) {
    // Dernier etage : I face a In compare Acc (1-s) a Fight (-2) ; face a Out : Fight (3 > 0).
    let rep_in: &str = if 1.0 - s > -2.0 { "Acc" } else { "Fight" };
    if rep_in == "Fight" {
        let g = gain(table, "Out", "Fight");
        return (("Out", "Fight"), (g.0 as f64, g.1 as f64));
    }
    (("In", rep_in), (2.0, 1.0 - s))
}

println!("{:>12} {:>16} {:>8}  regime", "s (caution)", "issue", "u_I");
for k in 0..9 {
    let s = k as f64 * 0.5;
    let (issue, pay) = equilibre_avec_bond(&u, s);
    let regime = if issue.1 == "Acc" { "annonce dissoute (non credible)" } else { "dissuasion tenue (credible)" };
    println!("{:>12.1} {:>16} {:>8.2}  {}", s, tup2s(issue.0, issue.1), pay.1, regime);
}
println!();
let (_, pay_3) = equilibre_avec_bond(&u, 3.0);
let ecart_tentation = u_i(&u, "In", "Acc") - u_i(&u, "In", "Fight");
println!("Seuil de credibilite : s* = {} (ecart de tentation 1 - (-2)).", ecart_tentation);
println!("Au seuil s = 3 : u_I = {} -- la pleine valeur d engagement est restauree.", pay_3.1);

 s (caution)            issue      u_I  regime


         0.0    ('In', 'Acc')     1.00  annonce dissoute (non credible)


         0.5    ('In', 'Acc')     0.50  annonce dissoute (non credible)


         1.0    ('In', 'Acc')     0.00  annonce dissoute (non credible)


         1.5    ('In', 'Acc')    -0.50  annonce dissoute (non credible)


         2.0    ('In', 'Acc')    -1.00  annonce dissoute (non credible)


... (2 outputs left out)

         3.5 ('Out', 'Fight')     3.00  dissuasion tenue (credible)


         4.0 ('Out', 'Fight')     3.00  dissuasion tenue (credible)


Seuil de credibilite : s* = 3 (ecart de tentation 1 - (-2)).


Au seuil s = 3 : u_I = 3 -- la pleine valeur d engagement est restauree.


### Le témoin exact : la « forme remarquable » calculée

La lecture de cette section argue, en prose, qu'il y a une « forme remarquable » : le seuil de caution `s*`, la pleine valeur d'engagement et l'écart de tentation **coïncident** (tombent tous sur `3`) et **dépassent** le dommage de la guerre (`2`). La cellule du seuil ci-dessus l'exhibe en **flottants** (balayage `0 → 4`).

Ce témoin recalcule ces grandeurs en **fractions exactes** et exhibe l'égalité et la stricte inégalité — le « remarquable » devient un résultat calculé, croisé aux flottants de la cellule du seuil, et non plus une argumentation de prose seule.

In [7]:

/// Type rationnel du carnet : numerateur / denominateur normalises, denominateur > 0.
#[derive(Clone, Copy, PartialEq, Eq)]
struct Rat { n: i64, d: i64 }

/// PGCD par l'algorithme d'Euclide.
fn pgcd(mut a: i64, mut b: i64) -> i64 {
    while b != 0 { let t = a % b; a = b; b = t; }
    a
}

impl Rat {
    fn new(n: i64, d: i64) -> Rat {
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = pgcd(n.abs(), d);
        Rat { n: n / g, d: d / g }
    }
    fn sub(&self, o: &Rat) -> Rat { Rat::new(self.n * o.d - o.n * self.d, self.d * o.d) }
    fn mul(&self, o: &Rat) -> Rat { Rat::new(self.n * o.n, self.d * o.d) }
    fn abs(&self) -> Rat { Rat::new(self.n.abs(), self.d) }
    fn eq_rat(&self, o: &Rat) -> bool { self.n * o.d == o.n * self.d }
    fn lt(&self, o: &Rat) -> bool { self.n * o.d < o.n * self.d }
    fn gt(&self, o: &Rat) -> bool { self.n * o.d > o.n * self.d }
    fn to_f64(&self) -> f64 { self.n as f64 / self.d as f64 }
}

impl std::fmt::Display for Rat {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        // `f.pad` : c'est lui qui applique la largeur et l'alignement demandes
        // par le format (`{: <10}`) ; un `write!` interne les perdrait.
        if self.d == 1 { f.pad(&self.n.to_string()) } else { f.pad(&format!("{}/{}", self.n, self.d)) }
    }
}
/// Booleen a la maniere de Python (True / False).
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Gains de l'incumbent (u_I) incrimines par le carnet, en fractions exactes :
let u_in_acc = Rat::new(1, 1);      // accommoder apres une entree
let u_in_fight = Rat::new(-2, 1);   // combattre apres une entree
let u_out_fight = Rat::new(3, 1);   // combattre, entrant dehors

// La « forme remarquable » en forme exacte :
let ecart_tentation = u_in_acc.sub(&u_in_fight);   // 1 - (-2)
let seuil_caution = ecart_tentation;              // s* : la caution couvre la tentation
let valeur_engagement = u_out_fight;              // pleine valeur d'engagement, restauree a s = s*
let dommage_guerre = u_in_fight.abs();            // dommage de la guerre

println!("La forme remarquable, en fractions exactes :");
println!("  ecart de tentation         = 1 - (-2)  = {}", ecart_tentation);
println!("  seuil de caution   s*      =           = {}", seuil_caution);
println!("  pleine valeur d'engagement =           = {}", valeur_engagement);
println!("  dommage de la guerre      = |(-2)|     = {}", dommage_guerre);
println!();
println!("  s* == valeur d'engagement : {}", py_bool(seuil_caution.eq_rat(&valeur_engagement)));
println!("  s*  >  dommage de guerre  : {}", py_bool(seuil_caution.gt(&dommage_guerre)));
println!();
println!("Balayage du seuil en forme exacte  (u_I = 1 - s sous le seuil, sinon la valeur d'engagement) :");
println!("   s    u_I exact   u_I flottant");
let flottants_carnet: [f64; 9] = [1.0, 0.5, 0.0, -0.5, -1.0, -1.5, 3.0, 3.0, 3.0];
let mut ecarts: Vec<f64> = Vec::new();
for k in 0..9 {
    let s = Rat::new(k as i64, 2);
    let u_i = if s.lt(&seuil_caution) { u_in_acc.sub(&s) } else { valeur_engagement };
    ecarts.push((u_i.to_f64() - flottants_carnet[k]).abs());
    println!("   {}   {:<10}  {:.2}", s, u_i, u_i.to_f64());
}
println!("\n  ecart max a l'exact sur le balayage : {:.6}", ecarts.iter().cloned().fold(f64::NEG_INFINITY, f64::max));

La forme remarquable, en fractions exactes :


  ecart de tentation         = 1 - (-2)  = 3


  seuil de caution   s*      =           = 3


  pleine valeur d'engagement =           = 3


  dommage de la guerre      = |(-2)|     = 2


  s* == valeur d'engagement : True


... (9 outputs left out)

   3   3           3.00


   7/2   3           3.00


   4   3           3.00


  ecart max a l'exact sur le balayage : 0.000000


### Lecture : le « remarquable » est exact, pas approché

Les trois grandeurs que la prose de cette section déclare « remarquables » se calculent égales **à la fraction près** : l'écart de tentation `1 - (-2)`, le seuil de caution `s*` et la pleine valeur d'engagement tombent tous sur `3`, et `3 > 2` — le dommage de la guerre — **strictement**. L'égalité `s* = 3 = valeur d'engagement` et la stricte inégalité `s* > dommage` sont donc des faits exacts, pas des rapprochements numériques.

Le balayage en fractions exactes recolle **exactement** aux flottants de la cellule du seuil (colonne `u_I flottant`) : `1, 1/2, 0, -1/2, -1, -3/2` sous le seuil, `3` au-delà — l'écart à l'exact est nul, ce qui confirme que les flottants affichés sont les décimales de ces rationnels.

### Lecture : le prix exact de la félicité — la caution doit couvrir la tentation

La caution transforme l'architecture : accommoder après avoir promis coûte désormais `1 − s`. Le balayage le montre avec une convention explicite à l’égalité : en dessous de `s* = 3`, l'entrant entre, l'incumbent accommode en payant la peine, et son gain **décroît de 1 − s** — le tableau le mesure ligne à ligne (0,50 à s = 0,5 ; 0,00 à s = 1 ; −1,50 à s = 2,5) : une caution sous le seuil est un coût pur, versée sans jamais acheter la dissuasion ; à partir de `s* = 3` (égalité exacte au seuil, stricte au-dessus), combattre devient la meilleure réponse **même face à une entrée réelle**, l'entrant le sait, reste dehors, et la valeur d'engagement **3** est intégralement restaurée. La condition de félicité d'Austin est devenue un nombre.

Et ce nombre a une forme remarquable : `s*` est exactement **l'écart de tentation** — la différence entre céder (1) et tenir (−2) au moment de la tester. Ce n'est ni le **delta** d'engagement (+2, mesuré la section correspondante : 3 avec engagement contre 1 sans engagement), ni le **dommage** de la guerre (2) : dans ce jeu, la caution à verser (3) **égale** la valeur d'engagement (3) — la « pleine valeur d'engagement » que la section correspondante restaure à `s = 3` — et **dépasse** le dommage de la guerre (2). Elle doit couvrir précisément ce que coûte le fait de dévier *au moment où dévier tenterait*. C'est la définition opérationnelle de la crédibilité : **crédible = dévier ne rapporte plus**, et le prix à payer pour y arriver se calcule sur la matrice. La « réputation », la « détermination », les métaphores volontaristes de la dissuasion se réduisent ici à cette inégalité.

À s = 3, Acc et Fight sont deux meilleures réponses après In. Le code sélectionne Fight au seuil ; une sélection Acc maintiendrait l’entrée. La dissuasion est strictement robuste pour s > 3. La pénalité change les gains ; ni trésorerie immobilisée ni prix financier de dépôt ne sont modélisés.



## Exercice 2 : le couple (engagement, gain) sur VOTRE matrice
<a id="engagement-mixte"></a>

Reprenez le duel publicitaire de l'Exercice 1. Construisez le tableau de l'engagement contraignant : pour **chacune** des deux actions pures de A, calculez la meilleure réponse de B et le gain de A qui en résulte. Produisez le **couple** `(stratégie d'engagement, gain)` optimal, et comparez-le au gain de A à l'issue séquentielle sans engagement — le delta d'engagement en un chiffre. Vérifiez enfin le mixte sur une grille `p ∈ [0, 1]` (pas de 0,05) : le sommet est-il pur, comme dans le jeu d'entrée, ou intérieur ?

La fonction étudiante retourne 21 entrées de la grille expérimentale (k = 0…20, p=k/20), chacune avec `k`, l’ensemble `br` des meilleures réponses de B, et les gains de A `optimiste`/`pessimiste`. Conserver les deux réponses quand B est indifférent. Le harnais vérifie chaque ligne sur plusieurs matrices puis calcule les meilleurs gains purs et mixtes, et leurs deltas contre B-premier. Les égalités du leader sont départagées par le premier ordre déclaré pour cette comparaison de référence.

```rust
# TODO etudiant
# Etape 1 : pour chaque action de A, BR de B puis gain de A.
# Etape 2 : argmax : couple engagement/gain et delta.
# Etape 3 : balayage mixte p par pas de 0.05, égalités et sommet intérieur.
```

In [8]:
type DuelCase = ((&'static str, &'static str), (i64, i64));

const AA: [&str; 2] = ["PasCampagne", "Campagne"];
const BB: [&str; 2] = ["Reste", "Attaque"];

/// Gain de la case (a, b), lu dans la matrice.
fn gain_d(m: &[DuelCase], a: &str, b: &str) -> (i64, i64) {
    m.iter().find(|(k, _)| k.0 == a && k.1 == b).map(|(_, v)| *v).expect("case de la matrice")
}

/// Matrice du duel publicitaire.
fn u_exo1() -> Vec<DuelCase> {
    vec![
        (("PasCampagne", "Reste"), (3, 3)),
        (("PasCampagne", "Attaque"), (1, 4)),
        (("Campagne", "Reste"), (2, 0)),
        (("Campagne", "Attaque"), (0, -1)),
    ]
}

/// Reference de B-premier : A repond par le max lexicographique (u_A, u_B), puis
/// toutes les issues de B maximales sont rendues.
fn reference_b_premier(matrice: &[DuelCase]) -> Vec<(&'static str, &'static str)> {
    let mut reponses: Vec<(&'static str, &'static str)> = Vec::new();
    for &b in BB.iter() {
        let mut meilleure = AA[0];
        let mut meilleur = gain_d(matrice, meilleure, b);
        for &a in AA.iter().skip(1) {
            let g = gain_d(matrice, a, b);
            if (g.0, g.1) > (meilleur.0, meilleur.1) { meilleure = a; meilleur = g; }
        }
        reponses.push((meilleure, b));
    }
    let cible = reponses.iter().map(|(a, b)| gain_d(matrice, *a, *b).1).fold(i64::MIN, i64::max);
    reponses.into_iter().filter(|(a, b)| gain_d(matrice, *a, *b).1 == cible).collect()
}
use std::collections::BTreeSet;

/// Une ligne de la grille experimentale du melange.
#[allow(dead_code)]
#[derive(Clone)]
struct LigneMixte {
    k: i64,
    br: Vec<&'static str>,
    optimiste: f64,
    pessimiste: f64,
}

fn engagements_duel(_matrice: &[DuelCase]) -> Option<Vec<LigneMixte>> {
    // TODO etudiant : liste de lignes k, br, optimiste, pessimiste.
    // Etape 1 : esperances par colonne ; Etape 2 : correspondance BR ; Etape 3 : les deux conventions.
    None
}

/// Format Python `{x:g}` : six chiffres significatifs, sans zero superflu.
fn py_g(x: f64) -> String {
    if x == x.trunc() && x.abs() < 1e16 {
        return format!("{}", x as i64);
    }
    let s = format!("{:.6}", x);
    let s = s.trim_end_matches('0');
    let s = s.trim_end_matches('.');
    s.to_string()
}

let base = u_exo1();
let f_plus: Vec<DuelCase> = base.iter().map(|((a, b), v)| {
    let bonus = if (*a, *b) == ("Campagne", "Reste") { 2 } else { 0 };
    ((*a, *b), (v.0, v.1 + bonus))
}).collect();

for matrice in [base.clone(), f_plus.clone()].iter() {
    match engagements_duel(matrice) {
        None => {
            println!("Exercice 2 à compléter : engagements purs, mixte, deltas");
            break;
        }
        Some(r) => {
            assert!(r.len() == 21, "la grille doit compter 21 lignes");
            let mut ks: Vec<i64> = r.iter().map(|l| l.k).collect();
            ks.sort_unstable();
            assert!(ks == (0..21).collect::<Vec<i64>>(), "les indices k doivent couvrir 0..20");
            for ligne in r.iter() {
                assert!((0..=20).contains(&ligne.k), "k hors bornes");
                let k = ligne.k;
                // Calcul entier du numerateur : l'egalite a p=1/2 reste exacte.
                let vb: Vec<(&'static str, i64)> = BB.iter().map(|b| {
                    (*b, (20 - k) * gain_d(matrice, AA[0], b).1 + k * gain_d(matrice, AA[1], b).1)
                }).collect();
                let maxi = vb.iter().map(|(_, v)| *v).fold(i64::MIN, i64::max);
                let attendu_br: BTreeSet<&'static str> = vb.iter().filter(|(_, v)| *v == maxi).map(|(b, _)| *b).collect();
                let obtenu_br: BTreeSet<&'static str> = ligne.br.iter().copied().collect();
                assert!(attendu_br == obtenu_br, "BR de B incorrect sur la ligne {}", k);
                let va: Vec<f64> = attendu_br.iter().map(|b| {
                    ((20 - k) * gain_d(matrice, AA[0], b).0 + k * gain_d(matrice, AA[1], b).0) as f64 / 20.0
                }).collect();
                let optimiste = va.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
                let pessimiste = va.iter().cloned().fold(f64::INFINITY, f64::min);
                assert!((ligne.optimiste - optimiste).abs() < 1e-10, "optimiste incorrect sur la ligne {}", k);
                assert!((ligne.pessimiste - pessimiste).abs() < 1e-10, "pessimiste incorrect sur la ligne {}", k);
            }
            let pur = r.iter().filter(|l| l.k == 0 || l.k == 20).map(|l| l.optimiste).fold(f64::NEG_INFINITY, f64::max);
            let mix = r.iter().map(|l| l.optimiste).fold(f64::NEG_INFINITY, f64::max);
            let base_issue = reference_b_premier(matrice);
            let base_gain = gain_d(matrice, base_issue[0].0, base_issue[0].1).0 as f64;
            println!("Exercice 2 : meilleur pur={}, meilleur mixte favorable={}, deltas={}/{}",
                     py_g(pur), py_g(mix), py_g(pur - base_gain), py_g(mix - base_gain));
        }
    }
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc.

Exercice 2 à compléter : engagements purs, mixte, deltas


## Le témoin de retenue : le dual du témoin d'obstruction

Les régimes précédents définissent l'agence par ce qu'un joueur **peut** faire : s'engager,
annoncer, cautionner. Il manque le revers : **un coup profitable à l'instant présent est
accessible, et le joueur ne le joue pas**. Cette comparaison locale ne suffit pas à établir
que le coup améliorerait son gain sur tout l'horizon :

```
pouvoir disponible  ≠  pouvoir exercé
```

Le piège : une abstention observée admet **trois causes indiscernables sur la seule trajectoire** —
la retenue locale (l'objet mesuré ici), un barème erroné (la déviation *croyait-on* profitable),
l'incapacité (l'action n'était pas atteignable). Un détecteur qui ne sépare pas les trois certifie
de la retenue partout où il y a une erreur de modèle. Le témoin ne vaut donc que s'il produit
d'abord le **certificat de disponibilité locale** : le coup énuméré dans l'ensemble d'actions
ET strictement améliorant pour **une période** au barème courant — *calculé et affiché* —
avant de constater qu'il n'a pas été joué.

Le dispositif : le duopole accommodant de la section 1, répété. L'entrant peut choisir une
guerre de prix éphémère (un « hit-and-run »), profitable sur la période du raid mais qui ferme
ensuite le marché pour lui. Son abstention atteste qu'il n'exerce pas ce pouvoir disponible ;
elle ne prouve pas un sacrifice de gain total. Le verdict se lit **depuis la trajectoire** —
l'historique des actions — jamais depuis une utilité déclarée.

In [9]:
/// Table du duopole : l'ordre de declaration est celui affiche par le temoin.
fn table_duopole() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("Raider", "Acc"), (5, -3)),
        (("Out", "Acc"), (0, 0)),
    ]
}

/// Gain de la case (ae, ai), lu dans la table.
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Repr Python d'un couple de chaines (apostrophes simples).
fn tup2s(a: &str, b: &str) -> String { format!("('{}', '{}')", a, b) }
/// Periode de duopole accommodant (definie comme dans la source ; non lue ici).
#[allow(dead_code)]
const HORIZON: usize = 10;

/// Gain total de l'entrant LU DEPUIS LA TRAJECTOIRE (suite d'actions, pas d'utilites).
#[allow(dead_code)]
fn gain_entrant(table: &[((&'static str, &'static str), (i64, i64))], histoire: &[(&'static str, &'static str)]) -> i64 {
    let mut total = 0;
    for (ae, ai) in histoire.iter() {
        total += gain(table, ae, ai).0;
    }
    total
}

/// Actions atteignables : les cles de la table, dans l'ordre declare (jamais trie).
fn actions_atteignables(table: &[((&'static str, &'static str), (i64, i64))]) -> Vec<(&'static str, &'static str)> {
    table.iter().map(|(k, _)| *k).collect()
}

let table = table_duopole();
let actions = actions_atteignables(&table);
let coup_raider = ("Raider", "Acc");
let gain_raider_1p = gain(&table, coup_raider.0, coup_raider.1).0;
let gain_in_1p = gain(&table, "In", "Acc").0;
let enumeration = format!("[{}]", actions.iter().map(|(a, b)| tup2s(a, b)).collect::<Vec<String>>().join(", "));

println!("Certificat de disponibilite du coup 'Raider' pour l'entrant :");
println!("  1. ENUMERATION : 'Raider' figure dans l'ensemble d'actions atteignables = {}", enumeration);
println!("  2. GAIN STRICT : raider une periode rapporte {} > cooperer {} (delta immediat = +{})",
         gain_raider_1p, gain_in_1p, gain_raider_1p - gain_in_1p);
println!("  3. La deviation est donc DISPONIBLE et STRICTEMENT PROFITABLE au bareme courant.");
println!();
println!("Le certificat pose, l'experience peut commencer : que fait l'entrant ?");

Certificat de disponibilite du coup 'Raider' pour l'entrant :


  1. ENUMERATION : 'Raider' figure dans l'ensemble d'actions atteignables = [('In', 'Acc'), ('Raider', 'Acc'), ('Out', 'Acc')]


  2. GAIN STRICT : raider une periode rapporte 5 > cooperer 2 (delta immediat = +3)


  3. La deviation est donc DISPONIBLE et STRICTEMENT PROFITABLE au bareme courant.


Le certificat pose, l'experience peut commencer : que fait l'entrant ?


**Lecture de la sortie committée** : le certificat tient en deux lignes vérifiables — l'action
est dans l'ensemble énuméré, et son gain **immédiat** (5 > 2) est calculé au barème courant,
pas déclaré. Il établit une occasion profitable sur une période, pas un gain supérieur sur les
dix périodes du jeu répété. La trajectoire suivante permet de mesurer séparément cet effet
à long terme.

In [10]:
/// Table du duopole : l'ordre de declaration est celui affiche par le temoin.
fn table_duopole() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("Raider", "Acc"), (5, -3)),
        (("Out", "Acc"), (0, 0)),
    ]
}

/// Gain de la case (ae, ai), lu dans la table.
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Booleen a la maniere de Python (True / False).
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Periode de duopole accommodant.
const HORIZON: usize = 10;

/// Gain total de l'entrant lu DEPUIS LA TRAJECTOIRE.
fn gain_entrant(table: &[((&'static str, &'static str), (i64, i64))], histoire: &[(&'static str, &'static str)]) -> i64 {
    let mut total = 0;
    for (ae, ai) in histoire.iter() {
        total += gain(table, ae, ai).0;
    }
    total
}

/// L'entrant raide a la periode 3, puis le marche se ferme pour lui (Out force).
fn trajectoire_exploit() -> Vec<(&'static str, &'static str)> {
    let mut h: Vec<(&'static str, &'static str)> = Vec::new();
    for _ in 0..2 { h.push(("In", "Acc")); }
    h.push(("Raider", "Acc"));
    for _ in 0..(HORIZON - 3) { h.push(("Out", "Acc")); }
    h
}

/// L'entrant NE joue PAS le coup disponible : duopole accommodant sur tout l'horizon.
fn trajectoire_retenue() -> Vec<(&'static str, &'static str)> {
    let mut h: Vec<(&'static str, &'static str)> = Vec::new();
    for _ in 0..HORIZON { h.push(("In", "Acc")); }
    h
}

/// Detecteur : le coup Raider apparait-il dans l'historique des ACTIONS ?
fn lire_raider(h: &[(&'static str, &'static str)]) -> bool {
    h.iter().any(|(ae, _)| *ae == "Raider")
}

/// Liste Python de chaines : ['In', 'Out', 'Raider'].
fn py_list_of_str(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<String>>().join(", "))
}

let table = table_duopole();
let trajectoires: Vec<(&str, Vec<(&'static str, &'static str)>)> = vec![
    ("trajectoire EXPLOIT", trajectoire_exploit()),
    ("trajectoire RETENUE", trajectoire_retenue()),
];
for (nom, h) in trajectoires.iter() {
    let lu = lire_raider(h);
    println!("{:22} : raider observe dans la trajectoire = {}", nom, py_bool(lu));
    println!("{:22}   gain total de l'entrant (lu, non declare) = {}", "", gain_entrant(&table, h));
    let mut distinctes: Vec<&str> = h.iter().map(|(ae, _)| *ae).collect();
    distinctes.sort_unstable();
    distinctes.dedup();
    println!("{:22}   actions distinctes jouees = {}", "", py_list_of_str(&distinctes));
}
println!();
println!("Divergence des deux histoires : MEME jeu, MEME certificat de disponibilite,");
println!("deux pouvoirs exercés différents — la retenue est le second.");

trajectoire EXPLOIT    : raider observe dans la trajectoire = True


                         gain total de l'entrant (lu, non declare) = 9


                         actions distinctes jouees = ['In', 'Out', 'Raider']


trajectoire RETENUE    : raider observe dans la trajectoire = False


                         gain total de l'entrant (lu, non declare) = 20


                         actions distinctes jouees = ['In']


Divergence des deux histoires : MEME jeu, MEME certificat de disponibilite,


deux pouvoirs exercés différents — la retenue est le second.


**Lecture de la sortie committée** : les deux trajectoires divergent sur un fait lisible —
la présence ou l'absence du raid dans **l'historique des actions**. Le gain total de l'entrant
est *calculé depuis* cet historique (jamais demandé au joueur) : l'exploit rapporte **9**
(deux périodes de duopole à 2, un raid à 5, puis sept périodes dehors à 0), contre **20**
(dix périodes à 2) lorsqu'il s'abstient. L'exploit rapporte donc **moins**, et non plus, sur
l'horizon de dix périodes. Le certificat précédent prouve seulement un avantage **immédiat**
(5 > 2 sur la période du raid) ; il ne certifie pas un avantage cumulé une fois la fermeture
du marché prise en compte. Ce témoin distingue un coup disponible non exercé d'un coup exercé,
mais ne démontre pas que l'abstention est sous-optimale sur l'horizon complet.

In [11]:
/// Table du duopole : l'ordre de declaration est celui affiche par le temoin.
fn table_duopole() -> Vec<((&'static str, &'static str), (i64, i64))> {
    vec![
        (("In", "Acc"), (2, 1)),
        (("Raider", "Acc"), (5, -3)),
        (("Out", "Acc"), (0, 0)),
    ]
}

/// Gain de la case (ae, ai), lu dans la table.
fn gain(table: &[((&'static str, &'static str), (i64, i64))], ae: &str, ai: &str) -> (i64, i64) {
    table.iter().find(|(k, _)| k.0 == ae && k.1 == ai).map(|(_, v)| *v).expect("case de la table")
}
/// Repr Python d'un tuple d'entiers.
fn tup2i(a: i64, b: i64) -> String { format!("({}, {})", a, b) }
/// Detecteur : le coup Raider apparait-il dans l'historique des ACTIONS ?
fn lire_raider(h: &[(&'static str, &'static str)]) -> bool {
    h.iter().any(|(ae, _)| *ae == "Raider")
}
use std::collections::BTreeSet;

/// Periode de duopole accommodant.
const HORIZON: usize = 10;

/// Un coup n'est DISPONIBLE que s'il est atteignable ET strictement ameliorant au bareme.
fn certificat_disponibilite(action: (&'static str, &'static str),
                             bareme: &[((&'static str, &'static str), (i64, i64))],
                             _br_actuelle: Option<&'static str>,
                             gain_actuel: i64) -> (bool, String) {
    match bareme.iter().find(|(k, _)| *k == action) {
        None => (false, "INCAPACITE : l'action n'est pas dans l'ensemble enumere".to_string()),
        Some((_, v)) => {
            let delta = v.0 - gain_actuel;
            if delta <= 0 {
                (false, format!("BAREME : le coup rapporte {} <= {} (non profitable)", v.0, gain_actuel))
            } else {
                (true, format!("DISPONIBLE : enumere, +{} strictement ameliorant", delta))
            }
        }
    }
}

let table = table_duopole();
let bareme_errone: Vec<((&'static str, &'static str), (i64, i64))> = vec![
    (("In", "Acc"), (2, 1)),
    (("Raider", "Acc"), (2, -3)),
    (("Out", "Acc"), (0, 0)),
];
let bareme_incapable: Vec<((&'static str, &'static str), (i64, i64))> = vec![
    (("In", "Acc"), (2, 1)),
    (("Out", "Acc"), (0, 0)),
];
let trajectoire_retenue: Vec<(&'static str, &'static str)> = vec![("In", "Acc"); HORIZON];

// (nom, action testee, bareme, gain actuel, histoire observee)
let cas: Vec<(&str, (&'static str, &'static str), Vec<((&'static str, &'static str), (i64, i64))>, i64, Vec<(&'static str, &'static str)>)> = vec![
    ("vraie retenue", ("Raider", "Acc"), table.clone(), 2, trajectoire_retenue.clone()),
    ("abstention par BAREME errone", ("Raider", "Acc"), bareme_errone.clone(), 2, trajectoire_retenue.clone()),
    ("abstention par INCAPACITE", ("Raider", "Acc"), bareme_incapable.clone(), 2, trajectoire_retenue.clone()),
];

let mut verdicts: Vec<(String, String)> = Vec::new();
for (nom, action, bareme, gain_actuel, histoire) in cas.iter() {
    let abstention = !lire_raider(histoire);
    let (dispo, motif) = certificat_disponibilite(*action, bareme, None, *gain_actuel);
    let verdict = if abstention && dispo {
        "RETENUE (abstention + coup disponible)".to_string()
    } else if abstention && !dispo {
        format!("ABSTENTION EXPLIQUEE SANS RETENUE : {}", motif)
    } else {
        "COUP JOUE (pas une abstention)".to_string()
    };
    verdicts.push((nom.to_string(), verdict.clone()));
    println!("--- {}", nom);
    let bareme_affiche = match bareme.iter().find(|(k, _)| *k == *action) {
        Some((_, v)) => tup2i(v.0, v.1),
        None => "ABSENT de l ensemble".to_string(),
    };
    println!("    bareme de 'Raider' : {}", bareme_affiche);
    println!("    certificat : {}", motif);
    println!("    verdict    : {}", verdict);
}

println!();
let uniques: BTreeSet<String> = verdicts.iter().map(|(_, v)| v.clone()).collect();
println!("Verdicts distincts sur les trois cas : {}/3", uniques.len());
println!("Un detecteur qui rendait le meme verdict trois fois serait rejete (acceptance #3).");

--- vraie retenue


    bareme de 'Raider' : (5, -3)


    certificat : DISPONIBLE : enumere, +3 strictement ameliorant


    verdict    : RETENUE (abstention + coup disponible)


--- abstention par BAREME errone


    bareme de 'Raider' : (2, -3)


... (4 outputs left out)

    certificat : INCAPACITE : l'action n'est pas dans l'ensemble enumere


    verdict    : ABSTENTION EXPLIQUEE SANS RETENUE : INCAPACITE : l'action n'est pas dans l'ensemble enumere


Verdicts distincts sur les trois cas : 3/3


Un detecteur qui rendait le meme verdict trois fois serait rejete (acceptance #3).


**Lecture de la sortie committée** : les trois cas rendent **trois verdicts distincts** — le
certificat sépare l'abstention malgré un avantage **immédiat**, le barème sans avantage immédiat
et l'incapacité. La « vraie retenue » dans ce contrôle exige la conjonction *abstention observée*
+ *coup énuméré* + *gain d'une période strictement supérieur au barème*. Ce verdict local ne
tranche pas si le coup aurait accru le gain cumulé : dans les trajectoires précédentes, le raid
fait précisément baisser celui-ci de 20 à 9. La caution de la section précédente rendait
l'annonce contraignante au prix exact ; ici, le certificat rend l'abstention significative à
l'échelle d'une décision, sans assimiler les deux échelles de gain.

**Ce que ce témoin n'est pas** : pas un score de coopération (coopérer sans avantage immédiat
ne satisfait pas ce certificat — voir le contrôle « barème » ; une action inaccessible échoue
au contrôle « incapacité ») ; pas une norme morale encodée, ni la preuve d'un sacrifice sur dix
périodes. Il mesure une action accessible non exercée, distincte de l'optimisation globale que
la sortie de trajectoire favorise ici.

## Résumé : quatre régimes, un seul nombre qui change, et ce qu'il prouve

| régime | mécanisme | issue | u_I |
|---|---|---|---|
| simultané, équilibre crédible | Nash `(In, Acc)` | entrée accueillie | 1 |
| simultané, équilibre de menace | Nash `(Out, Fight)` — incroyable | dissuasion *non tenable* | (3, fantôme) |
| séquentiel sans engagement | induction à rebours | `(In, Acc)` — la menace dissoute | 1 |
| **engagement contraignant** | `sigma_L -> BR_F(sigma_L)` | **`(Out, Fight)` — sélectionné** | **3** |
| annonce révocable | cheap talk — arbre à 3 étages | `(In, Acc)` — indépendant de l'annonce | 1 |
| engagement + caution s > 3 (ou s=3 avec tie Fight) | dévier ne rapporte plus | `(Out, Fight)` tenu sans jamais combattre | 3 |

Le noyau tient en trois phrases. **Dans cette matrice, l’engagement sélectionne un équilibre existant** que la menace seule ne pouvait pas tenir — le delta +2 était déjà dans la matrice, endormi dans la colonne `Out`. **L'acte engagé vaut par la modification de l'espace de réponse d'autrui**, non par son résultat direct : `Fight` vaut −2 *exécutée* contre une entrée, et 3 *commise* contre l'entrée qui n'aura pas lieu. **La félicité se mesure** : dans cet arbre, une annonce sans lien ne change pas l’issue, et restaurer le lien coûte exactement l'écart de tentation `s* = 3` — la caution minimale telle que dévier ne rapporte plus.

C'est une performativité sans mystère au sens strict : la chaîne `représentation publique -> anticipation collective -> changement des actions -> nouvelle réalité stratégique` est ici un calcul réversible, chaque flèche étant une cellule de ce notebook. Ce que le cadre ne couvre **pas** — et il faut l'écrire — : les contextes où l'engagement est multi-décisionnel et répété (réputation dynamique), où la caution est endogène (capacités détruites plutôt que caution versée), et les jeux où un engagement *mixte* bat le sommet pur (déjà illustré par l’exercice du duel publicitaire). Les autres extensions demandent un modèle supplémentaire — pas une extrapolation sociologique de celui-ci.

**Ouvertures de la série** : la sélection d'équilibre par contrainte rejoint les [jeux répétés et le folk theorem](Dynamics/Dynamics-02-Jeux-Repetes-Lean-Python.ipynb) (l'engagement comme cas limite de la punition crédible), et la modification de l'espace des actions plutôt que du choix en leur sein est exactement le geste de la strate 7 sur les vocabulaires — voir aussi [Metajeux-01, les open games](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb), où la représentation locale modifie le global dont elle est issue.


## Exercice 3 : l'évanouissement mesuré — annonce révocable et caution minimale
<a id="annonce-caution"></a>

Toujours sur le duel publicitaire : (a) codez l'arbre à trois étages « A annonce, B choisit, A choisit librement » et montrez par induction à rebours que l'issue est **indépendante de l'annonce** — le gain d'engagement s'évanouit ; (b) introduisez la caution `s` qui pénalise l'écart de promesse de A, balayez `s` et trouvez le seuil `s*` où l'engagement redevient crédible ; (c) vérifiez que `s*` égale l'écart de tentation de A sur la colonne critique. Sans (c), la félicité n'a pas été mesurée.

La fonction étudiante reçoit une matrice et s≥0. Retourner `reponses` (toutes les meilleures réponses de A après chaque choix de B, avec pénalité pour PasCampagne dans **les deux colonnes**), `issues` (B-premier, tie de A favorable à la promesse Campagne), `annonces` (les deux annonces révocables sans pénalité) et `seuils` (écarts de tentation non négatifs dans chacune des colonnes). Le contrat pénalise toute déviation de Campagne ; pénaliser une seule case peut laisser une déviation profitable ailleurs. Le contrôle fait varier s et les gains. Au seuil les réponses peuvent être multiples ; au-dessus du maximum des deux écarts, Campagne devient strictement optimale dans les deux sous-jeux.

```rust
# TODO etudiant
# Etape 1 : arbre à trois étages et induction à rebours.
# Etape 2 : caution s sur toute déviation, dans les deux colonnes.
# Etape 3 : s* contre les deux écarts de tentation ; ties au seuil.
```

In [12]:
type DuelCase = ((&'static str, &'static str), (i64, i64));

const AA: [&str; 2] = ["PasCampagne", "Campagne"];
const BB: [&str; 2] = ["Reste", "Attaque"];

/// Gain de la case (a, b), lu dans la matrice.
fn gain_d(m: &[DuelCase], a: &str, b: &str) -> (i64, i64) {
    m.iter().find(|(k, _)| k.0 == a && k.1 == b).map(|(_, v)| *v).expect("case de la matrice")
}

/// Matrice du duel publicitaire.
fn u_exo1() -> Vec<DuelCase> {
    vec![
        (("PasCampagne", "Reste"), (3, 3)),
        (("PasCampagne", "Attaque"), (1, 4)),
        (("Campagne", "Reste"), (2, 0)),
        (("Campagne", "Attaque"), (0, -1)),
    ]
}

/// Reference de B-premier : A repond par le max lexicographique (u_A, u_B), puis
/// toutes les issues de B maximales sont rendues.
fn reference_b_premier(matrice: &[DuelCase]) -> Vec<(&'static str, &'static str)> {
    let mut reponses: Vec<(&'static str, &'static str)> = Vec::new();
    for &b in BB.iter() {
        let mut meilleure = AA[0];
        let mut meilleur = gain_d(matrice, meilleure, b);
        for &a in AA.iter().skip(1) {
            let g = gain_d(matrice, a, b);
            if (g.0, g.1) > (meilleur.0, meilleur.1) { meilleure = a; meilleur = g; }
        }
        reponses.push((meilleure, b));
    }
    let cible = reponses.iter().map(|(a, b)| gain_d(matrice, *a, *b).1).fold(i64::MIN, i64::max);
    reponses.into_iter().filter(|(a, b)| gain_d(matrice, *a, *b).1 == cible).collect()
}
use std::collections::BTreeSet;

/// Contrat de retour de l'etudiant : reponses, issues, annonces, seuils.
#[allow(dead_code)]
#[derive(Clone)]
struct CautionRep {
    reponses: Vec<(&'static str, Vec<&'static str>)>,
    issues: Vec<(&'static str, &'static str)>,
    annonces: Vec<(&'static str, Vec<(&'static str, &'static str)>)>,
    seuils: Vec<(&'static str, f64)>,
}

fn annonce_caution_duel(_matrice: &[DuelCase], _s: f64) -> Option<CautionRep> {
    // TODO etudiant : reponses, issues, annonces (sans penalite), seuils.
    // Etape 1 : arbre a trois etages et induction a rebours.
    // Etape 2 : caution s sur toute deviation, dans les deux colonnes.
    // Etape 3 : s* contre les deux ecarts de tentation ; ties au seuil.
    None
}

fn verifier_caution_duel() {
    let base = u_exo1();
    let autres: Vec<DuelCase> = base.iter().map(|((a, b), v)| {
        let bonus = if (*a, *b) == ("PasCampagne", "Reste") { 3 } else { 0 };
        ((*a, *b), (v.0 + bonus, v.1))
    }).collect();
    for matrice in [base.clone(), autres.clone()].iter() {
        let annonces: Vec<(&'static str, Vec<(&'static str, &'static str)>)> =
            AA.iter().map(|a| (*a, reference_b_premier(matrice))).collect();
        let seuils: Vec<(&'static str, f64)> = BB.iter().map(|b| {
            (*b, (gain_d(matrice, AA[0], b).0 - gain_d(matrice, AA[1], b).0).max(0) as f64)
        }).collect();
        for s in [0.0f64, 0.5, 1.0, 1.5, 2.0, 4.0, 4.5].iter() {
            match annonce_caution_duel(matrice, *s) {
                None => {
                    println!("Exercice 3 à compléter : arbre révocable et pénalité de toute déviation");
                    return;
                }
                Some(r) => {
                    // La penalite s'applique a la deviation de la premiere action de A,
                    // dans LES DEUX colonnes.
                    let mut choisis: Vec<(&'static str, &'static str)> = Vec::new();
                    for &b in BB.iter() {
                        let gains: Vec<(&'static str, f64)> = AA.iter().map(|a| {
                            let penalite = if *a == AA[0] { *s } else { 0.0 };
                            (*a, gain_d(matrice, *a, b).0 as f64 - penalite)
                        }).collect();
                        let maxi = gains.iter().map(|(_, v)| *v).fold(f64::NEG_INFINITY, f64::max);
                        let attendu: BTreeSet<&'static str> = gains.iter()
                            .filter(|(_, v)| *v == maxi).map(|(a, _)| *a).collect();
                        let obtenu: BTreeSet<&'static str> = r.reponses.iter().find(|(k, _)| *k == b)
                            .map(|(_, v)| v.iter().copied().collect()).unwrap_or_default();
                        assert!(attendu == obtenu, "reponses incorrectes sur la colonne {}", b);
                        // Au seuil, le tie de A est departage en faveur de Campagne.
                        let choisi = if attendu.contains("Campagne") { "Campagne" }
                                     else { *attendu.iter().next().expect("reponse non vide") };
                        choisis.push((choisi, b));
                    }
                    let val = choisis.iter().map(|(a, b)| gain_d(matrice, *a, *b).1).fold(i64::MIN, i64::max);
                    let attendu_issues: BTreeSet<(&'static str, &'static str)> = choisis.iter()
                        .filter(|(a, b)| gain_d(matrice, *a, *b).1 == val).copied().collect();
                    let obtenu_issues: BTreeSet<(&'static str, &'static str)> = r.issues.iter().copied().collect();
                    assert!(attendu_issues == obtenu_issues, "issues incorrectes");
                    // Les annonces revocables ne portent aucune penalite.
                    for &a in AA.iter() {
                        let attendu: BTreeSet<(&'static str, &'static str)> = annonces.iter()
                            .find(|(k, _)| *k == a)
                            .map(|(_, v)| v.iter().copied().collect()).unwrap_or_default();
                        let obtenu: BTreeSet<(&'static str, &'static str)> = r.annonces.iter()
                            .find(|(k, _)| *k == a)
                            .map(|(_, v)| v.iter().copied().collect()).unwrap_or_default();
                        assert!(attendu == obtenu, "annonces incorrectes pour {}", a);
                    }
                    // Les seuils sont les ecarts de tentation, non negatifs, colonne par colonne.
                    for &b in BB.iter() {
                        let attendu = (gain_d(matrice, AA[0], b).0 - gain_d(matrice, AA[1], b).0).max(0) as f64;
                        let obtenu = r.seuils.iter().find(|(k, _)| *k == b).map(|(_, v)| *v).unwrap_or(f64::MIN);
                        assert!((obtenu - attendu).abs() < 1e-10, "seuil incorrect sur la colonne {}", b);
                    }
                }
            }
        }
    }
    println!("Exercice 3 : deux matrices, quatorze sous-jeux, ties et seuils validés");
}
verifier_caution_duel();

Exercice 3 à compléter : arbre révocable et pénalité de toute déviation


## À retenir

Trois activités restent ouvertes : [duel publicitaire](#duel-publicitaire), [engagement pur et mixte](#engagement-mixte), et [annonce/caution](#annonce-caution). Les contrôles n’affichent une validation qu’après une réponse fournie.

| Question | Objet calculé | Limite |
|---|---|---|
| Menace ou engagement ? | BR après choix contraignant | Information commune dans ce modèle |
| Annonce crédible ? | Gains de déviation et seuil de pénalité | Tie explicite au seuil |
| Retenue locale ? | Action accessible, gain immédiat, historique | Pas une preuve de sacrifice cumulé |

## Vérifiez votre compréhension

1. Pourquoi Fight n’est-elle pas globalement dominée ?
2. Pourquoi le seuil de caution n’est-il pas le delta d’engagement ?
3. Pourquoi l’abstention locale ne prouve-t-elle pas un sacrifice sur dix périodes ?

<details>
<summary>Éléments de réponse</summary>

1. Elle rapporte 3 contre Out, davantage que Acc ; son infériorité ne concerne que In.
2. Le seuil compare tenir/dévier dans un sous-jeu, le delta compare deux mécanismes initiaux.
3. Le raid rapporte 9 sur l’horizon contre 20 sans raid, malgré son avantage immédiat 5 contre 2.

</details>

## Pour aller plus loin

Von Stackelberg (1934), *Marktform und Gleichgewicht* ; Austin (1962), *How to Do Things with Words*. Le [carnet suivant](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) étudie les engagements de sécurité. Les liens de la série servent de points d’entrée ; aucune réplication empirique n’est promise ici.
